# Transfer Learning بموديل VGG16 على بيانات MNIST

هنستخدم موديل VGG16 المدرب مسبقاً على ImageNet، ونجمد طبقاته (Feature Extractor)، ونضيف فوقه طبقات Dense جديدة تتدرب على تصنيف أرقام MNIST.

## تحميل المكتبات وبيانات MNIST

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


## تجهيز الصور
VGG16 محتاج صور RGB (3 قنوات) ومقاس لا يقل عن 32x32، فهنحول الصور من رمادي لـ RGB ونكبرها لـ 32x32.

In [ ]:
def prepare(images):
    images = tf.expand_dims(images, -1)                     # (N,28,28) -> (N,28,28,1)
    images = tf.image.grayscale_to_rgb(images)               # -> (N,28,28,3)
    images = tf.image.resize(images, (32, 32))                # -> (N,32,32,3)
    images = preprocess_input(images)                         # تجهيز خاص بـ VGG16
    return images

x_train = prepare(x_train)
x_test = prepare(x_test)

print(x_train.shape, x_test.shape)


## تحميل VGG16 المدرب مسبقاً وتجميد طبقاته

In [ ]:
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False   # تجميد الطبقات، مش هتتدرب من جديد

base_model.summary()


## بناء الموديل الكامل (VGG16 + طبقات تصنيف جديدة)

In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


## التدريب
(بندرب بس الطبقات الجديدة، لأن VGG16 مجمد)

In [ ]:
history = model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1)


## التقييم على بيانات الاختبار

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)


## (اختياري) Fine-Tuning
لو عايز تحسن الدقة أكتر، ممكن تفك تجميد آخر كام طبقة من VGG16 وتدرب بمعدل تعلم صغير.

In [ ]:
base_model.trainable = True

# نجمد كل الطبقات ما عدا آخر 4 طبقات فقط
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_finetune = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1)


## التقييم بعد الـ Fine-Tuning

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy after fine-tuning:", test_acc)
